In [ ]:
building = 0
time = 24

# Setup

In [ ]:
import torch
import math
import pandas as pd
import numpy as np

import Project.src.data.dataprep as prep
import Project.src.data.featurisation as features

from Project.src.models.lstm import LSTM
from Project.src.models.lstmopt import LSTMOPT
from Project.src.models.lstmopt import CVXLayer

from Project.src.training.training import  Training as Training
from Project.src.training.training_opt import Training as Training_opt

import Project.src.optimization.pv_battery as pvb

import Project.src.tensors.tensorisation as tensor

import matplotlib.pyplot as plt

from torch.nn import functional as F

def torch_py(torch_tensor):
    return torch_tensor.cpu().detach().numpy().flatten()

def rescale(values, scaler):
    rescaled_values = values * (scaler[1] - scaler[0]) + scaler[0]   
    return rescaled_values

In [ ]:
# Import the base data and resample it from 5 minutes to hourly
nl_data = prep.dutch_data('../data/Dutchdata_clean/building_' + str(building) + '.parquet', 'h')
# Include net load for cost calculations
nl_data['net_load'] = nl_data['load'] - nl_data['solar_energy']
# Import a price profile and merge it with the consumption data
nl_price_data = pd.read_csv('../data/NL_DA_Prices.csv', index_col='Date',parse_dates=True, dayfirst=True)
nl_data = nl_data.merge(nl_price_data[['offtake', 'injection']], left_index=True, right_index=True)

# calculate the cost to be a positive net load x offtake minus a negative net load x injection
nl_data['cost'] = nl_data.apply(lambda row: row['net_load'] * row['injection'] if row['net_load'] < 0 else row['net_load'] * row['offtake'], axis=1)

In [ ]:
featurisation = features.Featurisation(nl_data)
nl_data = featurisation.cyclic_features(daily=False)[0]
nl_data.head()
features = 3

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parameters

In [ ]:
# Base parameters
battery_capacity = round(nl_data['solar_energy'].max() * 1.24,1)
max_charge = battery_capacity/2.7
max_discharge = max_charge
epochs = 100
layers = 3
neurons = 200

In [ ]:
pvb_system = pvb.PV_battery(nl_data, battery_capacity, max_charge, max_discharge)

In [ ]:
problem, variables, parameters = pvb_system.create_optimization_problem(24)

In [ ]:
# Tensors for training
nl_opt_tensors = tensor.Tensors(nl_data,'solar_energy',['solar_energy'],['month_sin','month_cos','load','offtake','injection'],24,24,forecast_gap=0, domain_min=[None, None, None, 0, 0, 0, None], domain_max=[None, None, None, 1, 1, 1, None])
X_train, X_test, y_train, y_test, scalers_opt = nl_opt_tensors.create_tensor()

# Previous problems

We first need the initial battery states for the hour prior to the current optimization as input for our current optimization. This requires looping over all previous timeslots, as every consecutive optimization needs the initial battery state coming from the previous one.

In [ ]:
# We evaluate how many times we need to loop over the optimization
problems = 24 - time

In [ ]:
# We keep the train initial batteries in this list
initial_battery_train_cvx = []

# We keep the test initial batteries in this list
initial_battery_test_cvx = []

# These values get updated each loop
old_T = 24          # The first timeslot under consideration: 24 hours to optimize 
lags = 24           # The number of lags for this first timeslot: the 24 previous hours
forecast_gap = 0    # The gap after every forecast which is 0 when we need to forecast 24 hours

# loop over each of the problems
for i in range(problems):
       
    print('Setting up optimization for hour ' + str(i))
    
    # Get the optimization problem for the current problem
    problem, variables, parameters = pvb_system.create_optimization_problem(old_T)
    
    # Get what actually happens based on the (dis)charging scheme to obtain the initial battery value for the next hour
    problem_post, variables_post, parameters_post = pvb_system.create_post_forecast_optimization_problem(old_T)

    
    # Tensors for training a base forecaster
    tensors = tensor.Tensors(pvb_system.house,'solar_energy',['solar_energy'],[],lags,old_T, forecast_gap=forecast_gap)
    X_train, X_test, y_train, y_test, scalers = tensors.create_tensor()
    
    
    # Tensors for training an E2E network
    tensors_opt = tensor.Tensors(pvb_system.house,'solar_energy',['solar_energy'],['load','offtake','injection'],lags,old_T, forecast_gap=forecast_gap, domain_min=[None, 0, 0, 0, None], domain_max=[None, 1, 1, 1, None])

    # We don't need the Y values as they are identical to the ones from the base forecaster
    X_train_opt_cvx, X_test_opt_cvx, _, _, scalers_opt_cvx = tensors_opt.create_tensor()
    
    # We have to assign initial battery values to the current optimization at hand, first we create an empty tensor
    initial_bat_tensor_train_cvx = torch.zeros([X_train_opt_cvx.shape[0],lags,1])
    initial_bat_tensor_test_cvx = torch.zeros([X_test_opt_cvx.shape[0],lags,1])
    
    # If this is the first optimization done at midnight, the initial battery is set at 50% of the capacity, as we also make sure that the end
    # state of the battery from the previous day is 50%
    if i == 0:       
        initial_bat_tensor_train_cvx[:,-1,:] = battery_capacity * 0.5
        initial_bat_tensor_test_cvx[:,-1,:] = battery_capacity * 0.5
        
    # If it is not the first optimization, we obtain the initial battery values from the list of battery values we have been maintaining 
    else:
        initial_bat_tensor_train_cvx[:,-1,:] = torch.tensor(initial_battery_train_cvx[i-1]).unsqueeze(-1)
        initial_bat_tensor_test_cvx[:,-1,:] = torch.tensor(initial_battery_test_cvx[i-1]).unsqueeze(-1) 
    
    # We add this tensor to our X tensors for the E2E network
    X_train_opt_cvx = torch.concat([X_train_opt_cvx, initial_bat_tensor_train_cvx],dim=-1)
    X_test_opt_cvx = torch.concat([X_test_opt_cvx, initial_bat_tensor_test_cvx],dim=-1)
       
    
    # Create the models for PV forecasts
    lstm_opt = LSTMOPT(1,neurons,layers,old_T,0.5,problem,parameters,variables,scalers_opt_cvx[0]).to(device)
    lstm_opt.load_state_dict(torch.load('../models/LSTM_CVX/building_' + str(building) + '_' + str(old_T) + 'h.pth'))
    
    print('Forecasting PV')
    
    # Forecast the PV
    pv_sol_train, _ = lstm_opt(X_train_opt_cvx[:,:,0].unsqueeze(-1).to(device), 
                         X_train_opt_cvx[:,-old_T:,1].to(device), 
                         X_train_opt_cvx[:,-old_T:,2].to(device), 
                         X_train_opt_cvx[:,-old_T:,3].to(device), 
                         X_train_opt_cvx[:,-1,4].to(device))
    
    pv_sol_test, _ = lstm_opt(X_test_opt_cvx[:,:,0].unsqueeze(-1).to(device), 
                         X_test_opt_cvx[:,-old_T:,1].to(device), 
                         X_test_opt_cvx[:,-old_T:,2].to(device), 
                         X_test_opt_cvx[:,-old_T:,3].to(device), 
                         X_test_opt_cvx[:,-1,4].to(device))
    
    
    # Make a list for the initial energy of the current timeslot for each day in the train and test sets
    lstm_train_initial_energy_t = []
    cvx_train_initial_energy_t = []
    
    lstm_test_initial_energy_t = []
    cvx_test_initial_energy_t = []
    
    print('Obtaining initial energy parameters for hour ' + str(i+1))
    
    # Loop over every day, first use the forecast of PV, next plug in the real PV and the charge and discharge schedules to obtain the real battery state
    for j in range(len(X_train)):           
        # CVX forecast
        parameters[0].value = torch_py(rescale(pv_sol_train[j], scalers_opt_cvx[0]))
        parameters[1].value = torch_py(X_train_opt_cvx[j,-old_T:,1])
        parameters[2].value = torch_py(X_train_opt_cvx[j,-old_T:,2])
        parameters[3].value = torch_py(X_train_opt_cvx[j,-old_T:,3])
        parameters[4].value = torch_py(X_train_opt_cvx[j,-1:,4])[0]
        problem.solve()
    
        # CVX real
        parameters_post[0].value = torch_py(rescale(y_train[j,:], scalers[0]))
        parameters_post[1].value = torch_py(X_train_opt_cvx[j,-old_T:,1])
        parameters_post[2].value = torch_py(X_train_opt_cvx[j,-old_T:,2])
        parameters_post[3].value = torch_py(X_train_opt_cvx[j,-old_T:,3])
        parameters_post[4].value = torch_py(X_train_opt_cvx[j,-1:,4])[0]
        parameters_post[5].value = variables[3].value
        parameters_post[6].value = variables[4].value    
        problem_post.solve()
        cvx_train_initial_energy_t.append(variables_post[2].value[1])
    
    # Do the same for the test set
    for j in range(len(X_test)):   
        # CVX forecast
        parameters[0].value = torch_py(rescale(pv_sol_test[j], scalers_opt_cvx[0]))
        parameters[1].value = torch_py(X_test_opt_cvx[j,-old_T:,1])
        parameters[2].value = torch_py(X_test_opt_cvx[j,-old_T:,2])
        parameters[3].value = torch_py(X_test_opt_cvx[j,-old_T:,3])
        parameters[4].value = torch_py(X_test_opt_cvx[j,-1:,4])[0]
        problem.solve()
    
        # CVX real
        parameters_post[0].value = torch_py(rescale(y_test[j,:], scalers[0]))
        parameters_post[1].value = torch_py(X_test_opt_cvx[j,-old_T:,1])
        parameters_post[2].value = torch_py(X_test_opt_cvx[j,-old_T:,2])
        parameters_post[3].value = torch_py(X_test_opt_cvx[j,-old_T:,3])
        parameters_post[4].value = torch_py(X_test_opt_cvx[j,-1:,4])[0]
        parameters_post[5].value = variables[3].value
        parameters_post[6].value = variables[4].value    
        problem_post.solve()
        cvx_test_initial_energy_t.append(variables_post[2].value[1])
    
    # Add the initial battery values to our list
    initial_battery_train_cvx.append(cvx_train_initial_energy_t)
    initial_battery_test_cvx.append(cvx_test_initial_energy_t)
    
    old_T += -1         # Update the timeslots we have to forecast, 1 less than the previous optimization
    lags += 1           # Update the lags we can use for the forecast, we still use the full previous day, but have 1 more hour of the current day as well
    forecast_gap += 1   # Add to the gap between forecasts (f.e. the gap is 1 if we only have to forecast 23 hours)

# LSTM model

First we make forecasts in a 2-step approach, for which we do not need the optimization.

## Tensors

In [ ]:
# Tensors for training
nl_tensors = tensor.Tensors(pvb_system.house,'solar_energy',['solar_energy'],['month_sin','month_cos'],lags,time,forecast_gap=forecast_gap)
X_train_base, X_test_base, y_train_base, y_test_base, scalers = nl_tensors.create_tensor()

## Model

In [ ]:
lstm = LSTM(features,neurons,layers,time,0.5).to(device)

In [ ]:
trainer_base = Training(lstm, X_train_base, y_train_base, X_test_base, y_test_base, epochs, learning_rate=0.0001)

In [ ]:
state_dicts_lstm, best_lstm = trainer_base.fit()

In [ ]:
lstm.load_state_dict(state_dicts_lstm[best_lstm])

# LSTM-CVX model

Next we make a model which integrates the optimization and the forecast. For this we need all parameters to do the optimization.

## Tensors

In [ ]:
# Tensors for training
nl_opt_tensors = tensor.Tensors(nl_data,'solar_energy',['solar_energy'],['month_sin', 'month_cos', 'load','offtake','injection'],lags,time,forecast_gap=forecast_gap, domain_min=[None, None, None, 0, 0, 0, None], domain_max=[None, None, None, 1, 1, 1, None])
X_train, X_test, _, _, scalers_opt = nl_opt_tensors.create_tensor()

In [ ]:
# Here we use the initial battery states which we obtained earlier from running the previous optimization programs

# Create an empty tensor to add to the X tensors
initial_bat_tensor_train = torch.zeros([X_train.shape[0],lags,1])
initial_bat_tensor_test = torch.zeros([X_test.shape[0],lags,1])

# If this is the first optimization done at midnight, the initial battery is set at 50% of the capacity, as we also make sure that the end
# state of the battery from the previous day is 50%
if problems == 0:
    initial_bat_tensor_train[:,-1,:] = battery_capacity * 0.5
    initial_bat_tensor_test[:,-1,:] = battery_capacity * 0.5

# If it is not the first optimization, we obtain the initial battery values from the list of battery values we have been maintaining 
else:
    initial_bat_tensor_train[:,-1,:] = torch.tensor(initial_battery_train_cvx[-1]).unsqueeze(-1)
    initial_bat_tensor_test[:,-1,:] = torch.tensor(initial_battery_test_cvx[-1]).unsqueeze(-1) 

X_train = torch.concat([X_train, initial_bat_tensor_train],dim=-1)
X_test = torch.concat([X_test, initial_bat_tensor_test],dim=-1)

In [ ]:
# Create the current optimization problem
problem, variables, parameters = pvb_system.create_optimization_problem(time)

In [ ]:
# Now we need to have optimal solutions given correct PV values in order to obtain the "regret" for training the E2E forecaster

# Create a CVXLayer (or just use a cvxpy solver, the result should be the same)
cvx_real = CVXLayer(problem, parameters, variables)

# get the optimal parameters for the train set
cvx_vars_train = cvx_real(rescale(y_train_base,scalers[-1]),
                          X_train[:,-time:,-4],
                          X_train[:,-time:,-3],
                          X_train[:,-time:,-2],
                          X_train[:,-1,-1])

# get the optimal parameters for the test set
cvx_vars_test = cvx_real(rescale(y_test_base,scalers[-1]),
                         X_test[:,-time:,-4],
                         X_test[:,-time:,-3],
                         X_test[:,-time:,-2],
                         X_test[:,-1,-1])

In [ ]:
"""
# Obtain the optimal outcomes: load x off-take - excess energy x injection
solution_train = (torch.bmm(cvx_vars_train[0].unsqueeze(1), X_train[:,-time:,-3].unsqueeze(-1)) -
                  torch.bmm(cvx_vars_train[1].unsqueeze(1), X_train[:,-time:,-2].unsqueeze(-1)))

solution_test = (torch.bmm(cvx_vars_test[0].unsqueeze(1), X_test[:,-time:,-3].unsqueeze(-1)) -
                 torch.bmm(cvx_vars_test[1].unsqueeze(1), X_test[:,-time:,-2].unsqueeze(-1)))
"""

In [ ]:
# Obtain the optimal outcomes: load x off-take - excess energy x injection
solution_train = torch.sub(torch.mul(cvx_vars_train[0], X_train[:,-time:,-3]),
                  torch.mul(cvx_vars_train[1], X_train[:,-time:,-2])).unsqueeze(-1)

solution_test = torch.sub(torch.mul(cvx_vars_test[0], X_test[:,-time:,-3]),
                 torch.mul(cvx_vars_test[1], X_test[:,-time:,-2])).unsqueeze(-1)

In [ ]:
'''
# Transform these solutions (which are single values per day) to the dimensionality of the problem
y_train_solution = F.pad(solution_train,(0,0,0,time-1))
y_test_solution = F.pad(solution_test,(0,0,0,time-1))
'''

In [ ]:
# Add the optimal costs to the Y tensor which now holds the correct PV values and the optimal costs, using the correct PV values
y_train_base = torch.cat([y_train_base.unsqueeze(-1), solution_train], dim=-1)
y_test_base = torch.cat([y_test_base.unsqueeze(-1), solution_test], dim=-1)

In [ ]:
# Create the post-forecast optimization problem which we need to calculate the regret
problem_post, variables_post, parameters_post = pvb_system.create_post_forecast_optimization_problem(time)
cvx_post = CVXLayer(problem_post, parameters_post, variables_post)

## Model

In [ ]:
# Make the model
lstm_opt = LSTMOPT(3,neurons,layers,time,0.5,problem,parameters,variables,scalers_opt[0]).to(device)

In [ ]:
lstm_opt.load_state_dict(state_dicts_lstm[best_lstm])

In [ ]:
# Train the model
trainer = Training_opt(lstm_opt, cvx_post, X_train, y_train_base, X_test, y_test_base, epochs, time, battery_capacity, min_beta=1, learning_rate=0.0001)

In [ ]:
state_dicts_cvx, best_cvx = trainer.fit(timer=True)

In [ ]:
lstm_opt.load_state_dict(state_dicts_cvx[best_cvx])

In [ ]:
torch.save(lstm.state_dict(), '../models/LSTM/building_' + str(building) + '_' + str(time) + 'h.pth')
torch.save(lstm_opt.state_dict(), '../models/LSTM_CVX/building_' + str(building) + '_' + str(time) + 'h.pth')